In [1]:
%pip install shapiq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import openml
import pandas as pd

# Load from OpenML ID 1461 (task 7592)
d = openml.datasets.get_dataset("bank-marketing", version=1)
X, y, cat_mask, attr_names = d.get_data(target=d.default_target_attribute)
X = pd.DataFrame(X, columns=attr_names[:-1])
y = pd.Series(y, name=d.default_target_attribute)

In [ ]:
import sys
import os
# In python file use __file__ , not notebook's directory 
notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..', 'custom_packages', 'bonXAI-main')))

In [ ]:
from bonXAI.core.tabular_preprocessor import TabularPreprocessor
prep = TabularPreprocessor(
    task_type="classification",
    id_like_threshold=0.99,
    scale_all_numeric_after_encoding=True,
    scale_target_in_regression=False,
    random_state=0,
)
X, y = prep.fit_transform(X, y)

In [5]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000, n_jobs=None) 
clf.fit(X, y)

LogisticRegression(max_iter=2000)

In [ ]:
from bonXAI.core.explainer import Explainer
exp = Explainer(model=clf, explainer_name="shapiq", strategy="kernel", seed=0)
vals_gt, t = exp.explain(
    X_foreground=X.values[:100],
    X_background=X.values,
    n_jobs=1
)
print(vals_gt.shape)           # -> (100, n_features); order 1 features - usual SHAP values
print(vals_gt)
pairs_gt = exp.shapiq_pairwise_  # list length 10; each item is a matrix or None
print(pairs_gt[0].shape)        # -> (n_features, n_features) or None; order 2 features - SHAP interaction values
print(pairs_gt)

Explaining with SHAPIQ (k-SII, max_order=2). 100 samples to explain using 45211 background samples.
(100, 15)
[[ 1.05252038e-03  1.01310963e-03  2.30652828e-03 ...  5.96011672e-03
  -5.73683995e-02 -2.03466469e-02]
 [-4.32081347e-04 -1.40793134e-03  3.11702945e-03 ...  5.55062004e-03
  -5.61772849e-02 -1.99225983e-02]
 [-1.68256099e-03  1.04113946e-04  3.79655080e-03 ... -3.18232072e-02
  -5.03461265e-02 -1.94390576e-02]
 ...
 [ 1.20879759e-03 -1.19601017e-03  2.83286187e-03 ...  5.72161018e-03
  -5.75212736e-02 -2.00825265e-02]
 [-6.30903930e-04 -1.22759994e-03  2.74306230e-03 ...  5.34368556e-03
  -5.51321741e-02 -1.97754676e-02]
 [ 6.50683095e-05 -9.70152939e-04  2.60368274e-03 ...  5.63674830e-03
  -5.65230574e-02 -2.00110430e-02]]
(15, 15)
[array([[ 0.00000000e+00, -2.83370368e-05, -8.98795432e-05,
        -7.19260626e-05,  5.15038033e-06,  8.14794794e-05,
         1.14476401e-04,  8.07500160e-05, -4.32080302e-05,
         6.79548235e-05,  2.24660883e-05, -1.30749619e-04,
        

### Pairs:
- A **list of length len(x_foreground)** (one per explained sample)
- **Each item** a **num(features) × num(features) symmetric matrix** of **order-2 (pairwise) interaction values**
- **Meaning** entry (i, j) is the **extra effect of features i and j together** beyond the sum of their individual main effects

In [ ]:
from bonXAI.core.metrics import topk_pair_overlap

vals, t = exp.explain(
    X_foreground=X.values[:100],
    X_background=X.values[:5],
    n_jobs=1
)
pairs = exp.shapiq_pairwise_

topk_score = topk_pair_overlap(pairs, pairs_gt, k=5)
topk_score

Explaining with SHAPIQ (k-SII, max_order=2). 100 samples to explain using 5 background samples.


1.0

In [15]:
pairs

[array([[ 0.00000000e+00, -2.83370368e-05, -8.98795432e-05,
         -7.19260626e-05,  5.15038033e-06,  8.14794794e-05,
          1.14476401e-04,  8.07500160e-05, -4.32080302e-05,
          6.79548235e-05,  2.24660883e-05, -1.30749619e-04,
          4.83920222e-05, -5.87765244e-04, -2.02452917e-04],
        [-2.83370368e-05,  0.00000000e+00,  6.32973256e-05,
         -4.08277722e-05, -8.85671345e-05,  1.83474705e-04,
          4.31270278e-05, -5.72400460e-05, -1.65836178e-04,
          4.89115188e-05,  5.81846139e-05,  1.06413563e-06,
          3.15992977e-05, -1.83250536e-04, -1.46644889e-04],
        [-8.98795432e-05,  6.32973256e-05,  0.00000000e+00,
         -5.03893817e-04, -3.36099933e-05, -2.35032906e-04,
         -6.75100427e-05,  1.29080826e-04,  1.55153184e-04,
          1.89233115e-04,  1.15932401e-04, -2.24913575e-04,
         -8.86389338e-05, -5.08819846e-04, -2.04568701e-04],
        [-7.19260626e-05, -4.08277722e-05, -5.03893817e-04,
          0.00000000e+00, -2.19842052

In [16]:
pairs_gt

[array([[ 0.00000000e+00, -2.83370368e-05, -8.98795432e-05,
         -7.19260626e-05,  5.15038033e-06,  8.14794794e-05,
          1.14476401e-04,  8.07500160e-05, -4.32080302e-05,
          6.79548235e-05,  2.24660883e-05, -1.30749619e-04,
          4.83920222e-05, -5.87765244e-04, -2.02452917e-04],
        [-2.83370368e-05,  0.00000000e+00,  6.32973256e-05,
         -4.08277722e-05, -8.85671345e-05,  1.83474705e-04,
          4.31270278e-05, -5.72400460e-05, -1.65836178e-04,
          4.89115188e-05,  5.81846139e-05,  1.06413563e-06,
          3.15992977e-05, -1.83250536e-04, -1.46644889e-04],
        [-8.98795432e-05,  6.32973256e-05,  0.00000000e+00,
         -5.03893817e-04, -3.36099933e-05, -2.35032906e-04,
         -6.75100427e-05,  1.29080826e-04,  1.55153184e-04,
          1.89233115e-04,  1.15932401e-04, -2.24913575e-04,
         -8.86389338e-05, -5.08819846e-04, -2.04568701e-04],
        [-7.19260626e-05, -4.08277722e-05, -5.03893817e-04,
          0.00000000e+00, -2.19842052

In [20]:
from bonXAI.core.metrics import compute_mmd 
mmd = 0
for i in range(100):
    mmd_i = compute_mmd(pairs[i], pairs_gt[i])
    mmd += mmd_i
mmd

0.0

In [21]:
from bonXAI.core.metrics import compute_mae
mae = 0
for i in range(100):
    mae_i = compute_mae(pairs[i], pairs_gt[i])
    mae += mae_i
mae

0.0

In [26]:
# not relevant for shapiq, just checking execution
from bonXAI.core.metrics import top_k_score
k_score = 0
for i in range(100):
    k_score_i = top_k_score(pairs[i], pairs_gt[i])
    k_score += k_score_i
print(k_score / 100)
print(top_k_score(pairs, pairs_gt))

1.0
1.0


### Hubert's implementation

In [4]:
import shapiq
X, y = shapiq.load_california_housing(to_numpy=True)
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X, y)

RandomForestRegressor()

In [5]:
explainer = shapiq.TabularExplainer(
    model=model,
    data=X
)

In [6]:
k = 512
X_background = X[:k]
imputer = shapiq.MarginalImputer(
    model=explainer.predict, 
    data=X_background, 
    sample_size=k
)

In [7]:
explainer = shapiq.TabularExplainer(
    model=model,
    data=X,
    approximator="regression",
    index="k-SII",
    max_order=2,
    imputer=imputer
)

In [10]:
interaction_values = explainer.explain(X[0], budget=1024)
print(interaction_values)

/opt/anaconda3/envs/cte/lib/python3.10/site-packages/shapiq/approximator/regression/base.py:152: UserWarning: Not all budget is required due to the border-trick.
  self._sampler.sample(budget)


InteractionValues(
    index=k-SII, max_order=2, min_order=0, estimated=False, estimation_budget=1024,
    n_players=8, baseline_value=1.9566147544921875,
    Top 10 interactions:
        (0,): 2.3025039212135168
        (): 1.9566147544921875
        (6,): 0.17647863125801402
        (0, 5): 0.08912473685583353
        (0, 2): 0.07420530640411344
        (2,): 0.058309707376080794
        (0, 1): 0.0462228920486963
        (0, 3): -0.04604776506821488
        (4,): -0.053111368638063684
        (0, 6): -0.2777277836877797
)


### new pachage imolementation

In [1]:
import shapiq
X, y = shapiq.load_california_housing(to_numpy=True)
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X, y)

RandomForestRegressor()

In [2]:
import sys
import os
# In python file use __file__ , not notebook's directory 
notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..', 'custom_packages', 'bonXAI-main')))

In [ ]:
from bonXAI.core.explainer import Explainer
exp = Explainer(model=model, task_type="regression", explainer_name="shapiq", strategy="na", seed=0)
vals_gt, t = exp.explain(
    X_foreground=X[:10],
    X_background=X[:20],
    n_jobs=12
)

Explaining with ShapIQ. 10 samples to explain using 20 background samples.


In [8]:
print(vals_gt[0].shape)
print(vals_gt)

(8, 8)
[array([[ 0.00000000e+00,  9.40228308e-03, -2.47099143e-02,
        -3.93509782e-02,  5.72366899e-02,  3.26837156e-01,
        -4.58224188e-02,  5.33408549e-03],
       [ 9.40228308e-03,  0.00000000e+00,  1.14338380e-02,
        -9.68852275e-03,  1.05659500e-02,  2.08428935e-02,
         8.43570055e-03,  4.95871063e-03],
       [-2.47099143e-02,  1.14338380e-02,  0.00000000e+00,
        -2.54129115e-03,  1.86554724e-02,  3.20030414e-02,
        -1.02295364e-02, -1.65239213e-03],
       [-3.93509782e-02, -9.68852275e-03, -2.54129115e-03,
         0.00000000e+00,  1.09565462e-03,  1.50385654e-02,
         5.45354732e-03,  1.66237765e-03],
       [ 5.72366899e-02,  1.05659500e-02,  1.86554724e-02,
         1.09565462e-03,  0.00000000e+00,  2.56121361e-02,
        -9.04318814e-04,  2.02678707e-04],
       [ 3.26837156e-01,  2.08428935e-02,  3.20030414e-02,
         1.50385654e-02,  2.56121361e-02,  0.00000000e+00,
         2.72729762e-02, -1.89954902e-02],
       [-4.58224188e-02,  